In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
gold_schema = "GOLD_tables"

fact_sales = spark.table(f"{catalog}.{gold_schema}.fact_sales")
dim_customer = spark.table(f"{catalog}.{gold_schema}.dim_customer")
dim_product = spark.table(f"{catalog}.{gold_schema}.dim_product")
dim_promotion = spark.table(f"{catalog}.{gold_schema}.dim_promotion")
dim_date = spark.table(f"{catalog}.{gold_schema}.dim_date")

In [0]:
kpi1_net_margin = (fact_sales
    .withColumn("gross_revenue", F.col("total_sales"))
    .withColumn("discount_amount", F.col("total_sales") * F.col("discount_applied"))
    .withColumn("net_margin", F.col("gross_revenue") - F.col("discount_amount"))
    .groupBy("store_location")
    .agg(
        F.sum("gross_revenue").alias("total_gross_revenue"),
        F.sum("discount_amount").alias("total_discounts"),
        F.sum("net_margin").alias("total_net_margin")
    )
    .orderBy(F.desc("total_net_margin")))

display(kpi1_net_margin)

store_location,total_gross_revenue,total_discounts,total_net_margin
Location D,1284255.54,281025.8398999999,1003229.7001000004
Location B,1241078.0100000014,278905.9536000003,962172.0563999995
Location C,1183512.3299999994,262051.86060000007,921460.4694000001
Location A,1141196.750000001,278178.4736000002,863018.2763999999
Unknown,675559.0300000003,136140.5427,539418.4872999999


In [0]:
kpi2_aov_by_promo = (fact_sales
    .join(dim_promotion, "promotion_sk", "left")
    .groupBy("promotion_type")
    .agg(
        F.avg("total_sales").alias("avg_order_value"),
        F.count("transaction_id").alias("num_transactions")
    )
    .orderBy(F.desc("avg_order_value")))

display(kpi2_aov_by_promo)


promotion_type,avg_order_value,num_transactions
Flash Sale,3074.0150284629985,527
Buy One Get One Free,2857.8895978062164,547
20% Off,2795.723928571429,560
Unknown,2122.1987158469956,366


In [0]:
kpi3_churn_heatmap = (dim_customer
    .filter("is_active = true")
    .groupBy("customer_state", "loyalty_program")
    .agg(
        F.count("customer_id").alias("total_customers"),
        F.sum(F.when(F.col("churned") == "Yes", 1).otherwise(0)).alias("churned_customers")
    )
    .withColumn("churn_rate_pct", F.round((F.col("churned_customers") / F.col("total_customers")) * 100, 2))
    .orderBy("customer_state", "loyalty_program"))

display(kpi3_churn_heatmap)

customer_state,loyalty_program,total_customers,churned_customers,churn_rate_pct
Old_State_1,No,1,0,0.0
Old_State_2,No,1,0,0.0
Old_State_3,No,1,0,0.0
State X,No,201,98,48.76
State X,Yes,161,89,55.28
State Y,No,164,71,43.29
State Y,Yes,180,92,51.11
State Z,No,180,85,47.22
State Z,Yes,161,84,52.17


In [0]:
dim_customer.filter(F.col("customer_state").like("Old_State%")).show(truncate=False)

+-----------+-----------+---+------+--------------+---------------+----------------+-------+--------------+------------------+---------------+-------------+-----------------+-------------+--------------+--------------------+------------------+---------+
|customer_sk|customer_id|age|gender|income_bracket|loyalty_program|membership_years|churned|marital_status|number_of_children|education_level|occupation   |customer_zip_code|customer_city|customer_state|effective_start_date|effective_end_date|is_active|
+-----------+-----------+---+------+--------------+---------------+----------------+-------+--------------+------------------+---------------+-------------+-----------------+-------------+--------------+--------------------+------------------+---------+
|1          |1          |56 |Other |High          |No             |0               |No     |Divorced      |3                 |Bachelor's     |Self-Employed|37848            |Old_City_1   |Old_State_1   |2026-07-10          |NULL          

In [0]:
kpi4_product_quality = (dim_product
    .filter("product_id != 'Unknown'")
    .groupBy("product_category")
    .agg(F.avg("product_return_rate").alias("avg_return_rate"), F.count("product_id").alias("num_products"))
    .orderBy(F.desc("avg_return_rate")))
display(kpi4_product_quality)

product_category,avg_return_rate,num_products
Electronics,0.2674033149171271,181
Furniture,0.26688679245283037,212
Groceries,0.2560180995475113,221
Clothing,0.23939534883720934,215
Toys,0.23292452830188698,212


In [0]:
kpi5_store_traffic = (fact_sales
    .groupBy("transaction_hour", "day_of_week")
    .agg(F.count("transaction_id").alias("num_transactions"))
    .orderBy(F.desc("num_transactions")))
display(kpi5_store_traffic)

transaction_hour,day_of_week,num_transactions
5,Wednesday,20
22,Thursday,19
5,Sunday,19
8,Thursday,18
7,Wednesday,18
22,Wednesday,17
20,Saturday,17
2,Sunday,17
1,Saturday,17
2,Saturday,17
